In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import urllib.request
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm

# ==========================================
# 1. 基础配置 (Configuration)
# ==========================================

class Config:
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0
        
        # 模型公共参数
        self.embed_dim = 64
        self.seq_len = 100 
        self.batch_size = 64
        self.epochs = 5
        self.lr = 0.001
        self.dropout = 0.3
        self.tcan_layers = 2
        self.kernel_size = 3
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        self.data_dir = "./data"
        self.dataset_name = "assistment-2009-2010-skill"
        self.fallback_url = "https://www.cs.wpi.edu/~gendong/data/assistments/skill_builder_data.csv"

# ==========================================
# 2. 数据处理与下载 (Data Processing)
# ==========================================

class DataProcessor:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def _prepare_data(self):
        """自动下载数据集"""
        csv_path = None
        for root, _, files in os.walk(self.config.data_dir):
            for file in files:
                if file.endswith(".csv") and "skill" in file.lower():
                    csv_path = os.path.join(root, file)
        
        if not csv_path:
            print(f"正在准备下载数据集: {self.config.dataset_name}")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
                from EduData import get_data
                get_data(self.config.dataset_name, self.config.data_dir)
            except:
                print("使用备用下载方案...")
                save_path = os.path.join(self.config.data_dir, "skill_builder_data.csv")
                urllib.request.urlretrieve(self.config.fallback_url, save_path)
            
            for root, _, files in os.walk(self.config.data_dir):
                for file in files:
                    if file.endswith(".csv") and "skill" in file.lower():
                        csv_path = os.path.join(root, file)
        return csv_path

    def process(self):
        file_path = self._prepare_data()
        df = pd.read_csv(file_path, encoding='latin-1', low_memory=False)
        df.dropna(subset=['skill_id'], inplace=True)
        df.drop_duplicates(subset=['user_id', 'problem_id', 'order_id'], inplace=True)

        # ID 映射
        user_map = {val: i+1 for i, val in enumerate(df['user_id'].unique())}
        prob_map = {val: i+1 for i, val in enumerate(df['problem_id'].unique())}
        skill_map = {val: i+1 for i, val in enumerate(df['skill_id'].unique())}
        
        df['uid'] = df['user_id'].map(user_map)
        df['iid'] = df['problem_id'].map(prob_map)
        df['sid'] = df['skill_id'].map(skill_map)
        
        self.config.num_students = len(user_map) + 1
        self.config.num_questions = len(prob_map) + 1
        self.config.num_concepts = len(skill_map) + 1

        # 计算题目难度 (Difficulty)
        item_diff = df.groupby('iid')['correct'].mean()
        diff_values = np.zeros(self.config.num_questions)
        for iid, diff in item_diff.items():
            diff_values[iid] = diff

        # 构建 Q-Matrix
        q_k_relation = np.zeros((self.config.num_questions, self.config.num_concepts))
        tmp_df = df[['iid', 'sid']].drop_duplicates()
        for _, row in tmp_df.iterrows():
            q_k_relation[int(row['iid']), int(row['sid'])] = 1
        # 归一化
        row_sums = q_k_relation.sum(axis=1)
        row_sums[row_sums == 0] = 1
        q_k_relation = q_k_relation / row_sums[:, np.newaxis]

        # 生成序列与记录
        sequences, records = [], []
        u_list = df['uid'].unique()[:2000] # 取2000名学生进行实验
        df_exp = df[df['uid'].isin(u_list)].sort_values('order_id')
        
        for uid, group in tqdm(df_exp.groupby('uid'), desc="数据预处理"):
            if len(group) < 5: continue
            sequences.append({'uid': uid, 'iids': group['iid'].values, 'labels': group['correct'].values})
            for _, row in group.iterrows():
                records.append([row['uid'], row['iid'], row['correct']])
        
        return sequences, records, q_k_relation, diff_values

# ==========================================
# 3. 完整版模型实现 (Full Models)
# ==========================================

class IRT(nn.Module):
    def __init__(self, n_user, n_item):
        super().__init__()
        self.theta = nn.Embedding(n_user, 1)
        self.beta = nn.Embedding(n_item, 1)
    def forward(self, uid, iid):
        return torch.sigmoid(self.theta(uid) - self.beta(iid)).squeeze()

class NeuralCDM(nn.Module):
    """完整版 NeuralCDM，包含单调性约束"""
    def __init__(self, n_user, n_item, n_skill, q_matrix):
        super().__init__()
        self.register_buffer('q_matrix', torch.tensor(q_matrix, dtype=torch.float32))
        self.student_emb = nn.Embedding(n_user, n_skill)
        self.item_emb = nn.Embedding(n_item, n_skill)
        self.layer1 = nn.Linear(n_skill, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 1)

    def forward(self, uid, iid):
        s = torch.sigmoid(self.student_emb(uid))
        i = torch.sigmoid(self.item_emb(iid))
        q = self.q_matrix[iid]
        x = (s - i) * q
        
        # 应用单调性约束: 权重必须非负
        x = F.linear(x, torch.abs(self.layer1.weight), self.layer1.bias)
        x = torch.sigmoid(x)
        x = F.linear(x, torch.abs(self.layer2.weight), self.layer2.bias)
        x = torch.sigmoid(x)
        x = F.linear(x, torch.abs(self.layer3.weight), self.layer3.bias)
        return torch.sigmoid(x).squeeze()

class HeteroGraphEmbedding(nn.Module):
    def __init__(self, config, q_k_relation, diff_values):
        super().__init__()
        self.q_emb = nn.Embedding(config.num_questions, config.embed_dim, padding_idx=0)
        self.k_emb = nn.Embedding(config.num_concepts, config.embed_dim, padding_idx=0)
        self.register_buffer('q_k_rel', torch.tensor(q_k_relation, dtype=torch.float32))
        self.register_buffer('diffs', torch.tensor(diff_values, dtype=torch.float32))
        self.diff_proj = nn.Linear(1, config.embed_dim)

    def forward(self, q_ids):
        # 知识点聚合
        k_agg = torch.matmul(self.q_k_rel, self.k_emb.weight)
        q_k_feat = F.embedding(q_ids, k_agg, padding_idx=0)
        # 难度嵌入
        d_feat = self.diff_proj(self.diffs[q_ids].unsqueeze(-1))
        # ID 嵌入
        q_id_feat = self.q_emb(q_ids)
        return q_id_feat + q_k_feat + d_feat

class TemporalAttention(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.q, self.k, self.v = nn.Linear(dim, dim), nn.Linear(dim, dim), nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, L, D = x.size()
        Q, K, V = self.q(x), self.k(x), self.v(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
        mask = torch.tril(torch.ones(L, L, device=x.device))
        scores = scores.masked_fill(mask == 0, -1e9)
        attn = self.dropout(F.softmax(scores, dim=-1))
        return torch.matmul(attn, V)

class HIG_TCAN_CD(nn.Module):
    def __init__(self, config, q_k_relation, diff_values):
        super().__init__()
        self.graph_emb = HeteroGraphEmbedding(config, q_k_relation, diff_values)
        self.student_emb = nn.Embedding(config.num_students, config.embed_dim, padding_idx=0)
        self.input_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)
        
        # TCAN Blocks
        self.tcan = nn.ModuleList([
            nn.Sequential(
                TemporalAttention(config.embed_dim),
                nn.Linear(config.embed_dim, config.embed_dim),
                nn.LayerNorm(config.embed_dim),
                nn.ReLU()
            ) for _ in range(config.tcan_layers)
        ])
        
        self.pred_mlp = nn.Sequential(
            nn.Linear(config.embed_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, q_ids, labels, u_ids):
        q_feat = self.graph_emb(q_ids)
        x = self.input_proj(torch.cat([q_feat, labels.unsqueeze(-1)], dim=-1))
        
        h = x
        for layer in self.tcan: h = layer(h)
        
        s_static = self.student_emb(u_ids).unsqueeze(1).expand(-1, q_ids.size(1), -1)
        
        # 对齐序列预测 (h_t, q_{t+1}, s)
        h_prev = h[:, :-1, :]
        q_next = q_feat[:, 1:, :]
        s_seq = s_static[:, 1:, :]
        
        feat = torch.cat([h_prev, q_next, s_seq], dim=-1)
        return self.pred_mlp(feat).squeeze(-1)

# ==========================================
# 4. 实验运行器 (Runner)
# ==========================================

class ExperimentRunner:
    def __init__(self, config):
        self.config = config
        self.results = {}

    def run(self, model_name, model, train_loader, test_loader, is_seq=False):
        model = model.to(self.config.device)
        optimizer = torch.optim.Adam(model.parameters(), lr=self.config.lr)
        
        print(f"\n>>> 正在开始训练: {model_name}")
        for epoch in range(self.config.epochs):
            model.train()
            total_loss = 0
            # 添加 Batch 进度条
            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{self.config.epochs}")
            for batch in pbar:
                optimizer.zero_grad()
                if is_seq:
                    q, r, u = [b.to(self.config.device) for b in batch]
                    preds = model(q, r, u)
                    loss = F.binary_cross_entropy(preds, r[:, 1:])
                else:
                    u, i, r = [b.to(self.config.device) for b in batch]
                    loss = F.binary_cross_entropy(model(u, i), r.float())
                
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
        # 评估
        model.eval()
        all_p, all_r = [], []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"评估 {model_name}"):
                if is_seq:
                    q, r, u = [b.to(self.config.device) for b in batch]
                    all_p.extend(model(q, r, u).cpu().numpy().flatten())
                    all_r.extend(r[:, 1:].cpu().numpy().flatten())
                else:
                    u, i, r = [b.to(self.config.device) for b in batch]
                    all_p.extend(model(u, i).cpu().numpy())
                    all_r.extend(r.cpu().numpy())
        
        auc = roc_auc_score(all_r, all_p)
        acc = accuracy_score(all_r, [1 if p>0.5 else 0 for p in all_p])
        self.results[model_name] = {'AUC': auc, 'ACC': acc}
        print(f"结果: {model_name} | AUC: {auc:.4f} | ACC: {acc:.4f}")

# ==========================================
# 5. 主程序
# ==========================================

def main():
    config = Config()
    processor = DataProcessor(config)
    
    # 数据加载
    seqs, recs, q_k_rel, diffs = processor.process()
    runner = ExperimentRunner(config)

    # 静态 DataLoader
    train_recs, test_recs = train_test_split(recs, test_size=0.2, random_state=42)
    static_train = DataLoader(train_recs, batch_size=config.batch_size, shuffle=True)
    static_test = DataLoader(test_recs, batch_size=config.batch_size)

    # 序列 DataLoader
    def collate_seq(batch):
        qs = [torch.tensor(x['iids'][:config.seq_len], dtype=torch.long) for x in batch]
        rs = [torch.tensor(x['labels'][:config.seq_len], dtype=torch.float32) for x in batch]
        us = torch.tensor([x['uid'] for x in batch], dtype=torch.long)
        
        # Padding
        qs_pad = torch.nn.utils.rnn.pad_sequence(qs, batch_first=True, padding_value=0)
        rs_pad = torch.nn.utils.rnn.pad_sequence(rs, batch_first=True, padding_value=0)
        
        # 截断或对齐到统一长度以便矩阵运算
        if qs_pad.size(1) < 2: # 异常情况处理
            qs_pad = torch.zeros((len(batch), 2), dtype=torch.long)
            rs_pad = torch.zeros((len(batch), 2))
            
        return qs_pad, rs_pad, us

    train_seqs, test_seqs = train_test_split(seqs, test_size=0.2, random_state=42)
    seq_train = DataLoader(train_seqs, batch_size=config.batch_size, collate_fn=collate_seq, shuffle=True)
    seq_test = DataLoader(test_seqs, batch_size=config.batch_size, collate_fn=collate_seq)

    # --- 开始实验 ---
    runner.run("IRT", IRT(config.num_students, config.num_questions), static_train, static_test)
    runner.run("NeuralCDM", NeuralCDM(config.num_students, config.num_questions, config.num_concepts, q_k_rel), static_train, static_test)
    runner.run("HIG-TCAN_CD", HIG_TCAN_CD(config, q_k_rel, diffs), seq_train, seq_test, is_seq=True)

    # --- 最终报表 ---
    print("\n" + "="*45)
    print(f"{'模型 (Model)':<18} | {'AUC':<10} | {'准确率 (ACC)':<10}")
    print("-" * 45)
    for model, metrics in runner.results.items():
        print(f"{model:<18} | {metrics['AUC']:.4f}     | {metrics['ACC']:.4f}")
    print("="*45)

if __name__ == "__main__":
    main()

正在准备下载数据集: assistment-2009-2010-skill
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.2/141.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 MB 27.6 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.4.4
    Uninstalling typeguard-4.4.4:
      Successfully uninstalled typeguard-4.4.4
  Attempting uninstall: filelock
    Found existing installation: filelock 3.20.1
    Uninstalling filelock-3.20.1:
      Successfully uninstalled filelock-3.20.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kauldron 1.3.0 requires typeguard>=4.4.1, but you have typeguard 4.1.2 which is incompatible.
ray 2.52.1 requires click!=8.3.*,>=7.0, but you have click 8.3.1 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
pytensor 2.35.1 requires filelock>=3.15, but you have filelock 3.11.0 which is incompatible.


downloader, INFO http://base.ustc.edu.cn/data/ASSISTment/2009_skill_builder_data_corrected.zip is saved as data/2009_skill_builder_data_corrected.zip


downloader, INFO data/2009_skill_builder_data_corrected.zip is unzip to data/2009_skill_builder_data_corrected


数据预处理: 100%|██████████| 2000/2000 [00:08<00:00, 224.71it/s] 


...

结果: IRT | AUC: 0.6517 | ACC: 0.6700

>>> 正在开始训练: NeuralCDM


评估 NeuralCDM: 100%|██████████| 689/689 [00:00<00:00, 1430.90it/s]


结果: NeuralCDM | AUC: 0.6327 | ACC: 0.6823

>>> 正在开始训练: HIG-TCAN_CD


评估 HIG-TCAN_CD: 100%|██████████| 6/6 [00:00<00:00, 260.17it/s]


结果: HIG-TCAN_CD | AUC: 0.9300 | ACC: 0.8539

模型 (Model)         | AUC        | 准确率 (ACC) 
---------------------------------------------
IRT                | 0.6517     | 0.6700
NeuralCDM          | 0.6327     | 0.6823
HIG-TCAN_CD        | 0.9300     | 0.8539
